# All Distribution Families in pymgcv

pymgcv supports **9 distribution families** with automatic smoothing parameter
selection. This notebook demonstrates each one with synthetic data.

| Family | Link | When to Use |
|--------|------|-------------|
| Gaussian | identity | Continuous, symmetric errors |
| Poisson | log | Count data |
| Binomial | logit | Binary / proportions |
| Gamma | log | Positive, right-skewed |
| Tweedie | log | Zero-inflated continuous |
| Negative Binomial | log | Over-dispersed counts |
| Inverse Gaussian | 1/μ² | Positive, heavy-tailed |
| Beta | logit | Proportions on (0,1) |
| Gaulss | identity | Location-scale regression |

In [ ]:
import numpy as np
import pandas as pd
from pymgcv import GAM, Tweedie

np.random.seed(42)
n = 500
x = np.random.uniform(0, 1, n)

## 1. Gaussian

In [ ]:
y = np.sin(2 * np.pi * x) + 0.3 * np.random.randn(n)
df = pd.DataFrame({"x": x, "y": y})

m = GAM("y ~ s(x)", data=df, family="gaussian", method="REML")
m.fit()
print(f"Gaussian — Deviance explained: {m.summary().split('Deviance explained')[0][-10:]}")
print(f"EDF: {list(m.edf_per_smooth.values())[0]['edf']:.2f}")

## 2. Poisson

In [ ]:
mu = np.exp(1 + 2 * np.sin(2 * np.pi * x))
y_pois = np.random.poisson(mu)
df = pd.DataFrame({"x": x, "y": y_pois})

m = GAM("y ~ s(x)", data=df, family="poisson", method="REML")
m.fit()
print(f"Poisson — EDF: {list(m.edf_per_smooth.values())[0]['edf']:.2f}")

## 3. Binomial

In [ ]:
from scipy.special import expit

p = expit(2 * np.sin(2 * np.pi * x))
y_bin = np.random.binomial(1, p)
df = pd.DataFrame({"x": x, "y": y_bin})

m = GAM("y ~ s(x)", data=df, family="binomial", method="REML")
m.fit()
print(f"Binomial — EDF: {list(m.edf_per_smooth.values())[0]['edf']:.2f}")

## 4. Gamma

In [ ]:
mu_gamma = np.exp(1 + 0.5 * np.sin(2 * np.pi * x))
y_gamma = np.random.gamma(shape=5, scale=mu_gamma / 5)
df = pd.DataFrame({"x": x, "y": y_gamma})

m = GAM("y ~ s(x)", data=df, family="gamma", method="REML")
m.fit()
print(f"Gamma — EDF: {list(m.edf_per_smooth.values())[0]['edf']:.2f}")

## 5. Tweedie (Fixed Power)

In [ ]:
# Use the campaign dataset for a realistic Tweedie example
df_tw = pd.read_csv("../campaign_data.csv")
df_tw["income_k"] = df_tw["income"] / 1000

m = GAM("spend ~ s(age) + s(income_k)", data=df_tw,
        family=Tweedie(power=1.5), method="REML")
m.fit()
print(f"Tweedie(p=1.5) — EDFs: {[v['edf'] for v in m.edf_per_smooth.values()]}")

## 6. Tweedie (Estimated Power)

In [ ]:
m = GAM("spend ~ s(age) + s(income_k)", data=df_tw,
        family=Tweedie(estimate_power=True), method="REML")
m.fit()
print(f"Tweedie(estimated) — p = {m.family.power:.3f}")
print(f"EDFs: {[v['edf'] for v in m.edf_per_smooth.values()]}")

## 7. Negative Binomial

In [ ]:
mu_nb = np.exp(1 + np.sin(2 * np.pi * x))
theta = 2.0  # dispersion
y_nb = np.random.negative_binomial(n=theta, p=theta / (theta + mu_nb))
df = pd.DataFrame({"x": x, "y": y_nb})

m = GAM("y ~ s(x)", data=df, family="negbinomial", method="REML")
m.fit()
print(f"NegBin — EDF: {list(m.edf_per_smooth.values())[0]['edf']:.2f}")

## Summary

All families follow the same API:
```python
model = GAM("y ~ s(x1) + s(x2)", data=df, family="<family>", method="REML")
model.fit()
```

The only difference is the `family` argument. pymgcv automatically selects:
- The correct **link function** (identity, log, logit, inverse)
- The correct **variance function** V(μ)
- The correct **working weights** and **working response** for PIRLS